### Cell 1: Import thư viện và các module đã tách

In [ ]:
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC

# Import từ các file của bạn
from config import *
from data_utils import *
from model_utils import *

### Cell 2: Đọc và Xử lý dữ liệu (ETL)

In [ ]:
# Đọc data thô
df_fake = pd.read_csv(RAW_FAKE_PATH); df_fake['label'] = 1
df_true = pd.read_csv(RAW_TRUE_PATH); df_true['label'] = 0

explore_raw_data(df_fake, "TẬP TIN GIẢ (FAKE.CSV)")
explore_raw_data(df_true, "TẬP TIN THẬT (TRUE.CSV)")

df_fake['date_parsed'] = pd.to_datetime(df_fake['date'], errors='coerce')
df_true['date_parsed'] = pd.to_datetime(df_true['date'], errors='coerce')

df_fake = df_fake.dropna(subset=['date_parsed']).sort_values('date_parsed').reset_index(drop=True)
df_true = df_true.dropna(subset=['date_parsed']).sort_values('date_parsed').reset_index(drop=True)

df_fake['is_test'] = False
df_fake.loc[int(len(df_fake) * 0.8):, 'is_test'] = True

df_true['is_test'] = False
df_true.loc[int(len(df_true) * 0.8):, 'is_test'] = True


df = pd.concat([df_fake, df_true], axis=0).reset_index(drop=True)
df = df.drop_duplicates(subset=['text']).dropna(subset=['text']).reset_index(drop=True)

df['text_clean_raw'] = df['text'].apply(deep_clean_text)
df['title_text_clean_raw'] = (df['title'] + " " + df['text']).apply(deep_clean_text)
df['title_only_clean_raw'] = df['title'].apply(deep_clean_text)
df['text_only_clean'] = df['text_clean_raw'].apply(lemmatize_and_remove_stopwords)
df['title_text_clean'] = df['title_text_clean_raw'].apply(lemmatize_and_remove_stopwords)
df['title_only_clean'] = df['title_only_clean_raw'].apply(lemmatize_and_remove_stopwords)

# Xóa trùng lặp sâu
df = df.drop_duplicates(subset=['text_only_clean']).reset_index(drop=True) # type: ignore
df = df[df['text_only_clean'].str.strip() != ""].reset_index(drop=True)

train_idx = df[df['is_test'] == False].index
test_idx = df[df['is_test'] == True].index
y_train = df.loc[train_idx, 'label']
y_test = df.loc[test_idx, 'label']

df.drop(columns=['subject', 'date', 'date_parsed', 'is_test'], inplace=True, errors='ignore')

explore_processed_data(df, train_idx, test_idx)

# Kiểm tra rò rỉ
check_data_leakage(df, train_idx, test_idx)

### Cell 3: Phân tích EDA

In [ ]:
plot_eda(df)

### Cell 4: Xây dựng Feature Pipeline

In [ ]:
X_train_text, X_test_text, tfidf_text = build_feature_pipeline(df, train_idx, test_idx, 'text_only_clean', 'text_clean_raw')
X_train_title, X_test_title, tfidf_title = build_feature_pipeline(df, train_idx, test_idx, 'title_text_clean', 'title_text_clean_raw')
X_train_title_only, X_test_title_only, tfidf_title_only = build_feature_pipeline(df, train_idx, test_idx, 'title_only_clean', 'title_only_clean_raw')

### Cell 5: Huấn luyện Mô hình

In [ ]:
all_results = []

# 1. Logistic Regression
lr_model = LogisticRegression(max_iter=1000, class_weight='balanced')
acc_lr_text, rep_lr_text = train_and_evaluate_model(lr_model, "LR (Text Only)", X_train_text, y_train, X_test_text, y_test, "Blues")
acc_lr_title, rep_lr_title = train_and_evaluate_model(lr_model, "LR (Title + Text)", X_train_title, y_train, X_test_title, y_test, "Blues")
acc_lr_title_only, rep_lr_title_only = train_and_evaluate_model(lr_model, "LR (Title Only)", X_train_title_only, y_train, X_test_title_only, y_test, "Blues")

display_comparison_report("LOGISTIC REGRESSION", acc_lr_title, rep_lr_title, acc_lr_text, rep_lr_text, acc_lr_title_only, rep_lr_title_only)

all_results.extend([
    ("Logistic Regression", "Text Only", acc_lr_text, rep_lr_text),
    ("Logistic Regression", "Title + Text", acc_lr_title, rep_lr_title),
    ("Logistic Regression", "Title Only", acc_lr_title_only, rep_lr_title_only)
])

# 2. Naive Bayes
print("--- HUẤN LUYỆN NAIVE BAYES ---")
nb_model = MultinomialNB()

acc_nb_text, rep_nb_text = train_and_evaluate_model(nb_model, "Naive Bayes (Text Only)", X_train_text, y_train, X_test_text, y_test, "Greens")
acc_nb_title, rep_nb_title = train_and_evaluate_model(nb_model, "Naive Bayes (Title + Text)", X_train_title, y_train, X_test_title, y_test, "Greens")
acc_nb_title_only, rep_nb_title_only = train_and_evaluate_model(nb_model, "Naive Bayes (Title Only)", X_train_title_only, y_train, X_test_title_only, y_test, "Greens")

display_comparison_report("NAIVE BAYES", acc_nb_title, rep_nb_title, acc_nb_text, rep_nb_text, acc_nb_title_only, rep_nb_title_only)

all_results.extend([
    ("Naive Bayes", "Text Only", acc_nb_text, rep_nb_text),
    ("Naive Bayes", "Title + Text", acc_nb_title, rep_nb_title),
    ("Naive Bayes", "Title Only", acc_nb_title_only, rep_nb_title_only)
])

# 3. SVM
print("--- HUẤN LUYỆN SVM ---")
svm_model = LinearSVC(random_state=RANDOM_STATE, max_iter=2000, class_weight='balanced')

acc_svm_text, rep_svm_text = train_and_evaluate_model(svm_model, "SVM (Text Only)", X_train_text, y_train, X_test_text, y_test, "Oranges")
acc_svm_title, rep_svm_title = train_and_evaluate_model(svm_model, "SVM (Title + Text)", X_train_title, y_train, X_test_title, y_test, "Oranges")
acc_svm_title_only, rep_svm_title_only = train_and_evaluate_model(svm_model, "SVM (Title Only)", X_train_title_only, y_train, X_test_title_only, y_test, "Oranges")

display_comparison_report("SVM", acc_svm_title, rep_svm_title, acc_svm_text, rep_svm_text, acc_svm_title_only, rep_svm_title_only)

all_results.extend([
    ("SVM", "Text Only", acc_svm_text, rep_svm_text),
    ("SVM", "Title + Text", acc_svm_title, rep_svm_title),
    ("SVM", "Title Only", acc_svm_title_only, rep_svm_title_only)
])

### Cell 6: Tổng hợp metric

In [ ]:
summary_df = generate_metrics_summary(all_results, METRICS_SUMMARY_PATH)

### Cell 7: WordCloud

In [ ]:
plot_wordclouds(df)

### Cell 8: Vẽ Top Features & Phân tích lỗ

In [ ]:
plot_top_features(tfidf_title, lr_model, "Logistic Regression (Title + Text)")

# Phân tích dự đoán sai cho mô hình tốt nhất
y_pred_best = lr_model.predict(X_test_title)
analyze_errors(df, test_idx, y_test, y_pred_best)